# 🇻🇳 Abstractive Summarization #1 — BARTpho + LoRA
### Kaggle T4 Optimized | OpenHust/vietnamese-summarization

**BARTpho** (`vinai/bartpho-word`) là model BART pre-trained trên tiếng Việt bởi VinAI.  
Kiến trúc encoder-decoder, sinh ra câu tóm tắt **mới**, không copy nguyên văn.


## 1. Cài đặt

In [4]:
!pip install -q transformers datasets evaluate rouge_score sentencepiece \
    accelerate huggingface_hub peft bitsandbytes

In [15]:
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.6 MB/s eta 0:00:00


In [5]:
import os, gc, re, json, warnings, logging
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
import psutil

warnings.filterwarnings("ignore")
SEED = 42
set_seed(SEED)

In [7]:
from google.colab import drive
import os

drive.mount('/content/drive')
save_dir = "/content/drive/MyDrive/NLP_Projects/bartpho-abs-summ"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


## 2. Cấu hình

In [8]:
CONFIG = {
    "model_name": "vinai/bartpho-word",

    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "v_proj"],

    "dataset_name": "OpenHust/vietnamese-summarization",
    "text_column": "Document",
    "summary_column": "Summary",

    "max_input_length": 512,
    "max_target_length": 128,

    "output_dir": "/content/drive/MyDrive/NLP_Projects/bartpho-abs-summ",

    "num_train_epochs": 3,

    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "lr_scheduler_type": "cosine",
    "fp16": True,
    "gradient_checkpointing": True,
    "optim": "adamw_torch_fused",

    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "rouge2",
    "greater_is_better": True,

    "early_stopping_patience": 1,

    "num_beams": 4,
    "no_repeat_ngram_size": 3,
    "length_penalty": 1.0,

    "test_size": 0.1,
    "val_size": 0.1,
}

## 3. Load Dataset

In [9]:
from huggingface_hub import HfApi, hf_hub_download

def load_openhust(name: str) -> Dataset:
    try:
        return load_dataset(name, split="train")
    except:
        pass

    api = HfApi()
    files = [f for f in api.list_repo_files(name, repo_type="dataset") if f.endswith(".csv")]
    dfs = []
    for f in files:
        try:
            path = hf_hub_download(repo_id=name, filename=f, repo_type="dataset")
            df = pd.read_csv(path)
            df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
            if {"Document", "Summary"}.issubset(df.columns):
                dfs.append(df)
                print(f"   ✓ {f}: {len(df):,}")
        except Exception as e:
            print(f"   ✗ {f}: {e}")

    merged = pd.concat(dfs, ignore_index=True).dropna(subset=["Document", "Summary"])
    merged = merged[(merged["Document"].str.len() > 10) & (merged["Summary"].str.len() > 5)]
    print(f"Total: {len(merged):,}")
    return Dataset.from_pandas(merged.reset_index(drop=True))

raw_dataset = load_openhust(CONFIG["dataset_name"])
print(f"   Columns: {raw_dataset.column_names}, Size: {len(raw_dataset):,}")

README.md: 0.00B [00:00, ?B/s]

Kmeans_1024_new.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Kmeans_512_new.csv:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

Kmeans_512_token_new.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

bio_medicine.csv:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

herding_512_bio_medicine.csv:   0%|          | 0.00/25.1M [00:00<?, ?B/s]

herding_bio_medicine.csv:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

herding_prompt_512_bio_medicine.csv:   0%|          | 0.00/29.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

   Columns: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'], Size: 74,564


In [10]:
print(len(raw_dataset))

74564


In [11]:
split1 = raw_dataset.train_test_split(test_size=0.1, seed=SEED)
split2 = split1["train"].train_test_split(test_size=0.1111, seed=SEED)

dataset_dict = DatasetDict({
    "train": split2["train"].select(range(min(15000, len(split2["train"])))),
    "validation": split2["test"].select(range(min(1500, len(split2["test"])))),
    "test": split1["test"].select(range(min(1500, len(split1["test"])))),
})

for name, ds in dataset_dict.items():
    print(f"   {name}: {len(ds):,}")

   train: 15,000
   validation: 1,500
   test: 1,500


## 4. Tokenizer & Preprocessing

In [16]:
from pyvi import ViTokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
print(f"Tokenizer vocab: {tokenizer.vocab_size:,}")

def preprocess(examples):
    inputs = [ViTokenizer.tokenize(re.sub(r'\s+', ' ', str(d)).strip()) for d in examples[CONFIG["text_column"]]]
    targets = [ViTokenizer.tokenize(re.sub(r'\s+', ' ', str(s)).strip()) for s in examples[CONFIG["summary_column"]]]
    model_inputs = tokenizer(
        inputs,
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=CONFIG["max_target_length"],
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized = dataset_dict.map(
    preprocess, batched=True, batch_size=1000,
    remove_columns=dataset_dict["train"].column_names,
    desc="Tokenizing",
)


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer vocab: 64,000
Tokenizing...


Tokenizing:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1500 [00:00<?, ? examples/s]

## 5. Load BARTpho + LoRA

In [13]:
print(f"Loading {CONFIG['model_name']}...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

if CONFIG["gradient_checkpointing"]:
    base_model.gradient_checkpointing_enable()
    if hasattr(base_model, "enable_input_require_grads"):
        base_model.enable_input_require_grads()
    else:
        def _hook(module, input, output):
            output.requires_grad_(True)
        base_model.get_input_embeddings().register_forward_hook(_hook)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading vinai/bartpho-word...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 2,359,296 || all params: 553,794,560 || trainable%: 0.4260


## 6. Training

In [17]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    result["gen_len"] = np.mean([len(p.split()) for p in decoded_preds])
    return {k: round(v, 4) for k, v in result.items()}


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    fp16=CONFIG["fp16"],
    gradient_checkpointing=CONFIG["gradient_checkpointing"],
    optim=CONFIG["optim"],
    eval_strategy=CONFIG["eval_strategy"],
    save_strategy=CONFIG["save_strategy"],
    save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=CONFIG["load_best_model_at_end"],
    metric_for_best_model=CONFIG["metric_for_best_model"],
    greater_is_better=CONFIG["greater_is_better"],
    predict_with_generate=True,
    generation_max_length=CONFIG["max_target_length"],
    generation_num_beams=CONFIG["num_beams"],
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)

eff = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

In [18]:
print("Training BARTpho + LoRA...")

result = trainer.train()

print(f"\n Done! Loss: {result.training_loss:.4f}, "
      f"Time: {result.metrics.get('train_runtime',0)/60:.1f}m")

Training BARTpho + LoRA...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,9.944321,2.218602,0.368600,0.176800,0.254600,0.254500,92.672700
2,9.354595,2.147139,0.359000,0.175700,0.249000,0.249100,97.592000



 Done! Loss: 9.9397, Time: 82.4m


In [21]:
save_path = os.path.join(CONFIG["output_dir"], "best-lora")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f" Saved: {save_path}")

 Saved: /content/drive/MyDrive/NLP_Projects/bartpho-abs-summ/best-lora


## 7. Đánh giá Test Set

In [19]:
test_results = trainer.evaluate(eval_dataset=tokenized["test"], metric_key_prefix="test")

print(f"   ROUGE-1:    {test_results.get('test_rouge1', 0):.4f}")
print(f"   ROUGE-2:    {test_results.get('test_rouge2', 0):.4f}")
print(f"   ROUGE-L:    {test_results.get('test_rougeL', 0):.4f}")
print(f"   ROUGE-Lsum: {test_results.get('test_rougeLsum', 0):.4f}")
print(f"   Avg GenLen: {test_results.get('test_gen_len', 0):.1f}")

early stopping required metric_for_best_model, but did not find eval_rouge2 so early stopping is disabled


   ROUGE-1:    0.3551
   ROUGE-2:    0.1735
   ROUGE-L:    0.2456
   ROUGE-Lsum: 0.2455
   Avg GenLen: 96.3


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 8. Inference & Demo

In [23]:
@torch.no_grad()
def summarize(text: str, max_length: int = 256, num_beams: int = 4) -> str:
    """Abstractive summarization với BARTpho + LoRA."""
    model.eval()
    inputs = tokenizer(
        re.sub(r'\s+', ' ', text).strip(),
        max_length=CONFIG["max_input_length"],
        truncation=True,
        return_tensors="pt",
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        no_repeat_ngram_size=CONFIG["no_repeat_ngram_size"],
        length_penalty=CONFIG["length_penalty"],
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# Demo
test_data = dataset_dict["test"]
for i in range(min(5, len(test_data))):
    doc = test_data[i][CONFIG["text_column"]]
    ref = test_data[i][CONFIG["summary_column"]]
    pred = summarize(doc)

    r = rouge.compute(predictions=[pred], references=[ref])

    print(f"\n{'─'*70}")
    print(f"#{i+1}")
    print(f"   Doc:  {doc[:200]}...")
    print(f"   Ref:  {ref}")
    print(f"   Pred: {pred}")
    print(f"   R1={r['rouge1']:.3f} R2={r['rouge2']:.3f} RL={r['rougeL']:.3f}")


──────────────────────────────────────────────────────────────────────
#1
   Doc:  Bộ Giáo dục đào tạo vừa gửi công văn đến các sở GD-ĐT , các trường ĐH , các học viện và các trường cao đẳng sư phạm , trung cấp sư phạm trên toàn quốc ngày 28-12. Đó là tình trạng " thiếu trung thực t...
   Ref:  Bộ Giáo dục đào tạo nhắc lại lời kêu gọi chống tiêu cực trong một chỉ thị cách đây 11 năm , nhấn mạnh chống bệnh thành tích một cách thường xuyên , không phô trương , hình thức .
   Pred: Bộ Giáo dục đào tạo vừa gửi công văn đến các sở , các trường ĐH , các học viện và các trường cao đẳng sư phạm , trung cấp sư phạm trên toàn quốc ngày 28-1212 về tình trạng " thiếu trung thực trong học tập , giảng dạy , nghiên cứu khoa học , thi và kiểm tra " .
   R1=0.500 R2=0.173 RL=0.316

──────────────────────────────────────────────────────────────────────
#2
   Doc:  Thời gian gần đây , việc mâu thuẫn , cạnh tranh trong làm ăn liên tục khiến dư luận phải bức xúc .Hết chuyện đổ dầu luyn vào hàng thịt lợn 

In [24]:
custom = """Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến
toàn quốc về phát triển công nghệ trí tuệ nhân tạo (AI). Thủ tướng nhấn mạnh Việt Nam
cần nắm bắt cơ hội từ cuộc cách mạng AI để phát triển kinh tế - xã hội, đồng thời xây
dựng hành lang pháp lý phù hợp. Theo báo cáo, thị trường AI tại Việt Nam đang tăng
trưởng mạnh với hơn 500 doanh nghiệp hoạt động trong lĩnh vực này. Bộ Khoa học và
Công nghệ đã trình Chiến lược quốc gia về AI đến năm 2030."""

print(f" Input: {custom[:200]}...")
print(f"\n Summary: {summarize(custom)}")

 Input: Sáng 15/3, tại Hà Nội, Thủ tướng Chính phủ đã chủ trì Hội nghị trực tuyến
toàn quốc về phát triển công nghệ trí tuệ nhân tạo (AI). Thủ tướng nhấn mạnh Việt Nam
cần nắm bắt cơ hội từ cuộc cách mạng AI ...

 Summary: Việt_Nam cần nắm bắt cơ hội từ cuộc cách mạng AI để phát triển kinh tế - xã hội​ , đồng thời xây dựng hành lang pháp lý phù hợp.
